# Multi-Task v4 Kendall — 5-Fold CV: **Eval Only**

Decoupled from `multitask_v4_5fold_cv.ipynb` (the training notebook).

- **Re-uses** every def from the train notebook by exec'ing its cells (skipping only the actual training loop).
- **Caches per-fold OOF predictions** to `RUN_DIR/fold_*/oof_*.npz` so post-hoc analysis (per-class dice, Cobb-from-seg, plots) is pure-numpy and instant.
- **Loads models once** for cross-fold ensemble + TTA (these need full forwards on every case).

Workflow:
1. Train once using the training notebook (5h). Walk away.
2. Run this notebook anytime — re-runs in seconds after the first OOF dump.

In [1]:
# === Bootstrap: exec all setup/def cells from the train notebook ===
import json
import sys
from pathlib import Path

TRAIN_NB = Path("multitask_v4_5fold_cv.ipynb")
assert TRAIN_NB.exists(), f"missing {TRAIN_NB.resolve()}"

# Cells to skip when bootstrapping (these are training-time only):
#   cell 22 — the actual `for fold_idx, ...: train_one_fold(...)` loop
SKIP_CELLS = {22}

_nb = json.loads(TRAIN_NB.read_text())
for _i, _c in enumerate(_nb["cells"]):
    if _c["cell_type"] != "code":
        continue
    if _i in SKIP_CELLS:
        print(f"  skip cell {_i} (train loop)")
        continue
    exec("".join(_c["source"]), globals())

print(f"\nbootstrap done — RUN_DIR={RUN_DIR.relative_to(REPO_ROOT)}")

repo=/home/ortiz/scoliosis
device=privateuseone:0
seed=42
size=512x256  seg_classes=18  vertebrae=17  keypoints=68
total=250  trainable=249 (cobb_gt=178)
K-fold splits (5 folds, n_total=249):
  fold 0: train=199  val=50 (scoliosis_val=32, cobb_gt_val=32)
  fold 1: train=199  val=50 (scoliosis_val=39, cobb_gt_val=39)
  fold 2: train=199  val=50 (scoliosis_val=35, cobb_gt_val=35)
  fold 3: train=199  val=50 (scoliosis_val=35, cobb_gt_val=35)
  fold 4: train=200  val=49 (scoliosis_val=37, cobb_gt_val=37)
  image           (4, 1, 512, 256)  torch.float32
  seg             (4, 512, 256)  torch.int64
  kpt_heatmap     (4, 68, 128, 64)  torch.float32
  kpt_valid       (4, 68)  torch.bool
  cobb            (4,)  torch.float32
  cobb_valid      (4,)  torch.bool


Dropped Escape call with ulEscapeCode : 0x03007703


MultiTaskEncoderUNet total params: 24,501,303
  encoder: 21,278,400  decoder+heads: 3,222,903
run dir: ai/models/checkpoints/multitask_v4_5fold/20260503T1715_249cfc78  hash=249cfc78
  skip cell 22 (train loop)


NameError: name 'RESULTS_DF' is not defined

In [ ]:
# === Verify all fold checkpoints are on disk ===
FOLD_DIRS = [RUN_DIR / f"fold_{k}" for k in range(N_FOLDS)]
VARIANT_CKPTS = {
    "best_dice": "model.pt",
    "best_cobb": "model_best_cobb.pt",
    "ema":       "model_ema.pt",
}

missing = []
for fd in FOLD_DIRS:
    for v, name in VARIANT_CKPTS.items():
        if not (fd / name).exists():
            missing.append(f"{fd.name}/{name}")
if missing:
    print("MISSING ckpts (training incomplete or wrong RUN_DIR):")
    for m in missing: print("  ", m)
else:
    print("all fold ckpts present")

for fd in FOLD_DIRS:
    m = json.loads((fd / "metrics.json").read_text())
    print(f"  {fd.name}: dice={m['best_val_dice']:.3f}@ep{m['best_val_dice_epoch']}  "
          f"cobb={m['best_val_cobb_mae']:.2f}°@ep{m['best_val_cobb_mae_epoch']}  "
          f"stopped@{m['stopped_epoch']}")

## OOF cache — per-fold own-val predictions

Builds (if missing) `fold_{k}/oof_{variant}.npz` containing:
- `seg_logits` `(N_val, C, H, W)` float16
- `seg_gt`     `(N_val, H, W)`     uint8
- `cobb_pred`  `(N_val,)`          float32
- `cobb_gt`    `(N_val,)`          float32 (NaN where no GT)
- `case_ids`   `(N_val,)`          str

First run takes ~30s/fold/variant. Subsequent runs skip if the npz exists.

In [ ]:
import numpy as np
import torch
from datetime import datetime

@torch.no_grad()
def _forward_val(model, val_df):
    model.eval()
    seg_logits, seg_gt, cobb_preds, gts, ids = [], [], [], [], []
    for _, row in val_df.iterrows():
        case = preprocess_case(row)
        img = case["image"].unsqueeze(0).to(DEVICE)
        out = model(img)
        seg_logits.append(out["seg"].squeeze(0).half().cpu().numpy())
        seg_gt.append(case["seg"].numpy().astype(np.uint8))
        cobb_preds.append(float(out["cobb"].squeeze(0).cpu()))
        gts.append(float(row["cobb_angle_deg"]) if pd.notna(row.get("cobb_angle_deg")) else np.nan)
        ids.append(str(row.get("id_paciente", row.name)))
    return {
        "seg_logits": np.stack(seg_logits, axis=0),
        "seg_gt":     np.stack(seg_gt, axis=0),
        "cobb_pred":  np.asarray(cobb_preds, dtype=np.float32),
        "cobb_gt":    np.asarray(gts, dtype=np.float32),
        "case_ids":   np.asarray(ids),
    }

def dump_fold_oof(fold_idx, val_idx, variants=("best_dice", "best_cobb", "ema"), force=False):
    fd = RUN_DIR / f"fold_{fold_idx}"
    val_df = TRAINABLE.iloc[val_idx].reset_index(drop=True)
    for v in variants:
        out_path = fd / f"oof_{v}.npz"
        if out_path.exists() and not force:
            print(f"  fold {fold_idx} {v:10s}: cached ({out_path.stat().st_size/1e6:.1f} MB)")
            continue
        ckpt_path = fd / VARIANT_CKPTS[v]
        if not ckpt_path.exists():
            print(f"  fold {fold_idx} {v:10s}: ckpt missing, skip")
            continue
        m = MultiTaskEncoderUNet(pretrained=False, dropout=DROPOUT).to(DEVICE)
        state = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        m.load_state_dict(state["model"])
        oof = _forward_val(m, val_df)
        np.savez_compressed(out_path, **oof)
        print(f"  fold {fold_idx} {v:10s}: wrote {out_path.name} ({out_path.stat().st_size/1e6:.1f} MB)")
        del m
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

for fold_idx, (_, val_idx) in enumerate(FOLD_SPLITS):
    dump_fold_oof(fold_idx, val_idx)

manifest = {
    "config_hash": cfg_hash(CFG),
    "config": CFG,
    "fold_val_indices": {str(k): val_idx.tolist() for k, (_, val_idx) in enumerate(FOLD_SPLITS)},
    "saved_at": datetime.utcnow().isoformat() + "Z",
}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
print(f"\nmanifest: {(RUN_DIR/'manifest.json').relative_to(REPO_ROOT)}")

## Per-fold metrics from cache (pure numpy)

Re-derives dice + Cobb MAE per fold from the cached OOF npz. No model loading.

In [ ]:
def load_oof(fold_idx, variant):
    p = RUN_DIR / f"fold_{fold_idx}" / f"oof_{variant}.npz"
    if not p.exists():
        return None
    z = np.load(p, allow_pickle=False)
    return {k: z[k] for k in z.files}

def dice_from_oof(oof, num_classes=NUM_SEG_CLASSES):
    pred = oof["seg_logits"].astype(np.float32).argmax(axis=1)
    gt   = oof["seg_gt"]
    inter = np.zeros(num_classes - 1, dtype=np.float64)
    card  = np.zeros(num_classes - 1, dtype=np.float64)
    for c in range(1, num_classes):
        p_c = (pred == c)
        g_c = (gt == c)
        inter[c - 1] = (p_c & g_c).sum()
        card[c - 1]  = p_c.sum() + g_c.sum()
    valid = card > 0
    per_class = np.where(valid, (2.0 * inter) / np.clip(card, 1e-6, None), np.nan)
    return float(np.nanmean(per_class[valid])) if valid.any() else float("nan"), per_class

def cobb_mae_from_oof(oof):
    mask = ~np.isnan(oof["cobb_gt"])
    if not mask.any():
        return float("nan")
    return float(np.mean(np.abs(oof["cobb_pred"][mask] - oof["cobb_gt"][mask])))

rows = []
for v in VARIANT_CKPTS:
    for k in range(N_FOLDS):
        oof = load_oof(k, v)
        if oof is None:
            continue
        d, _ = dice_from_oof(oof)
        c    = cobb_mae_from_oof(oof)
        rows.append({"variant": v, "fold": k, "n_val": len(oof["case_ids"]), "dice": d, "cobb_mae": c})
OOF_DF = pd.DataFrame(rows)
OOF_DF.to_csv(RUN_DIR / "per_fold_oof.csv", index=False)
print(OOF_DF.to_string(index=False))

## Aggregate — mean ± SD across folds

Headline thesis numbers.

In [ ]:
agg = (OOF_DF
       .groupby("variant")
       .agg(dice_mean=("dice", "mean"), dice_std=("dice", "std"),
            dice_min=("dice", "min"),  dice_max=("dice", "max"),
            cobb_mean=("cobb_mae", "mean"), cobb_std=("cobb_mae", "std"),
            cobb_min=("cobb_mae", "min"),  cobb_max=("cobb_mae", "max"))
       .reset_index())
agg.to_csv(RUN_DIR / "aggregate_oof.csv", index=False)

for _, r in agg.iterrows():
    print(f"=== {r['variant']} ===")
    print(f"  dice:     {r['dice_mean']:.3f} ± {r['dice_std']:.3f}   range [{r['dice_min']:.3f}, {r['dice_max']:.3f}]")
    print(f"  cobb_mae: {r['cobb_mean']:.2f}° ± {r['cobb_std']:.2f}°   range [{r['cobb_min']:.2f}°, {r['cobb_max']:.2f}°]")
    print()

## Cross-fold ensemble inference (streamed — VRAM-safe)

Average predictions from all K fold-models on the pooled val set (each case appears in exactly one fold's val).

**Caveat**: each fold's model was trained on the OTHER folds' val cases, so this is mildly inflated vs a held-out test. Wiki flags this as **leaked — do not report** (`[[2026-05-04_v4_5fold_cv]]`). Kept only as sanity check.

**Streaming**: loads ONE model on GPU at a time, dumps logits to CPU, frees GPU, loads next. Avoids the 5-models-on-GPU OOM crash that killed the kernel.

**Memory budget**:
- GPU: 1 model (~50 MB) + 1 forward at a time
- CPU: accumulator `(N, C, H, W)` float32 ≈ 1.4 GB RAM for N=150

Slower (~3–5 min for 3 variants) but won't crash. **Skip this cell entirely if you only need thesis numbers** — those come from the OOF cells above.

In [ ]:
import gc

def _vram_free_gb() -> float:
    if not torch.cuda.is_available():
        return float("inf")
    free, _ = torch.cuda.mem_get_info()
    return free / 1e9


@torch.no_grad()
def evaluate_kfold_ensemble_streamed(fold_dirs, val_df, ckpt_name="model.pt"):
    """Memory-safe ensemble: one model on GPU at a time, sums on CPU."""
    K = len(fold_dirs)
    N = len(val_df)

    # Pre-cache GT once (CPU). Same across folds.
    cases = [preprocess_case(row) for _, row in val_df.iterrows()]
    seg_gt_cache  = np.stack([c["seg"].numpy().astype(np.uint8) for c in cases], axis=0)
    cobb_gt_cache = np.full(N, np.nan, dtype=np.float32)
    for i, (_, row) in enumerate(val_df.iterrows()):
        gt = row.get("cobb_angle_deg")
        if pd.notna(gt):
            cobb_gt_cache[i] = float(gt)

    seg_logits_sum = None
    cobb_pred_sum  = np.zeros(N, dtype=np.float64)
    n_models_used  = 0

    for fi, fd in enumerate(fold_dirs):
        ckpt_path = fd / ckpt_name
        if not ckpt_path.exists():
            print(f"  fold {fi}: missing {ckpt_name}, skip")
            continue
        m = MultiTaskEncoderUNet(pretrained=False, dropout=DROPOUT).to(DEVICE)
        state = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        m.load_state_dict(state["model"])
        m.eval()

        for i, c in enumerate(cases):
            img = c["image"].unsqueeze(0).to(DEVICE)
            out = m(img)
            seg_np = out["seg"].squeeze(0).float().cpu().numpy()  # (C, H, W) → CPU
            if seg_logits_sum is None:
                seg_logits_sum = np.zeros((N, *seg_np.shape), dtype=np.float32)
            seg_logits_sum[i] += seg_np
            cobb_pred_sum[i]  += float(out["cobb"].squeeze(0).cpu())

        n_models_used += 1
        del m, state
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"  fold {fi}: done  (free VRAM: {_vram_free_gb():.2f} GB)")

    if seg_logits_sum is None or n_models_used == 0:
        return {"dice": float("nan"), "cobb_mae": float("nan"), "n_models": 0}

    seg_pred  = (seg_logits_sum / n_models_used).argmax(axis=1)  # (N, H, W)
    cobb_pred = cobb_pred_sum / n_models_used

    inter = np.zeros(NUM_SEG_CLASSES - 1, dtype=np.float64)
    card  = np.zeros(NUM_SEG_CLASSES - 1, dtype=np.float64)
    for c in range(1, NUM_SEG_CLASSES):
        p = (seg_pred == c)
        g = (seg_gt_cache == c)
        inter[c - 1] = (p & g).sum()
        card[c - 1]  = p.sum() + g.sum()
    valid = card > 0
    dice = (
        float(np.where(valid, (2.0 * inter) / np.clip(card, 1e-6, None), np.nan)[valid].mean())
        if valid.any() else float("nan")
    )

    mask = ~np.isnan(cobb_gt_cache)
    cobb_mae = float(np.mean(np.abs(cobb_pred[mask] - cobb_gt_cache[mask]))) if mask.any() else float("nan")
    return {"dice": dice, "cobb_mae": cobb_mae, "n_models": n_models_used}


# Defensive cleanup before the VRAM-heavy run
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"VRAM free at start: {_vram_free_gb():.2f} GB")

all_val_idx = np.concatenate([va for _, va in FOLD_SPLITS])
ALL_VAL_DF = TRAINABLE.iloc[all_val_idx].reset_index(drop=True)

ENSEMBLE_RESULTS = {}
for v, name in VARIANT_CKPTS.items():
    if not all((fd / name).exists() for fd in FOLD_DIRS):
        print(f"=== {v}: ckpts missing, skip ===")
        continue
    print(f"\n=== K-Fold Ensemble (streamed) — {v} ckpts ===")
    res = evaluate_kfold_ensemble_streamed(FOLD_DIRS, ALL_VAL_DF, ckpt_name=name)
    ENSEMBLE_RESULTS[v] = res
    print(f"  dice = {res['dice']:.3f}")
    print(f"  cobb_direct MAE = {res['cobb_mae']:.2f}°")

(RUN_DIR / "ensemble_results.json").write_text(json.dumps(ENSEMBLE_RESULTS, indent=2))
print(f"\nsaved: {(RUN_DIR/'ensemble_results.json').relative_to(REPO_ROOT)}")

## TTA per-fold (hflip-only)

Horizontal flip preserves `|Cobb|` → free 2× ensemble at inference. Average seg logits before argmax; average `cobb_direct`.

In [ ]:
@torch.no_grad()
def evaluate_tta_per_fold(fold_dirs, val_dfs, ckpt_name="model.pt"):
    rows = []
    for k, (fd, val_df) in enumerate(zip(fold_dirs, val_dfs)):
        m = MultiTaskEncoderUNet(pretrained=False, dropout=DROPOUT).to(DEVICE)
        state = torch.load(fd / ckpt_name, map_location="cpu", weights_only=False)
        m.load_state_dict(state["model"])
        m.eval()
        inter = torch.zeros(NUM_SEG_CLASSES - 1, device=DEVICE)
        card  = torch.zeros(NUM_SEG_CLASSES - 1, device=DEVICE)
        cobb_abs_err = []
        for _, row in val_df.iterrows():
            case = preprocess_case(row)
            img = case["image"].unsqueeze(0).to(DEVICE)
            img_flipped = torch.flip(img, dims=[-1])
            out = m(img)
            out_flip = m(img_flipped)
            seg_avg = (out["seg"] + torch.flip(out_flip["seg"], dims=[-1])) / 2.0
            cobb_avg = (out["cobb"].squeeze(0).cpu().item() + out_flip["cobb"].squeeze(0).cpu().item()) / 2.0
            seg_pred = seg_avg.argmax(dim=1).squeeze(0)
            seg_t = case["seg"].to(DEVICE)
            for c in range(1, NUM_SEG_CLASSES):
                p, g = (seg_pred == c), (seg_t == c)
                inter[c - 1] += (p & g).sum()
                card[c - 1]  += p.sum() + g.sum()
            gt = row.get("cobb_angle_deg")
            if pd.notna(gt):
                cobb_abs_err.append(abs(cobb_avg - float(gt)))
        dice_per_cls = (2.0 * inter) / card.clamp(min=1e-6)
        valid = card > 0
        rows.append({
            "fold": k,
            "dice_tta":     float(dice_per_cls[valid].mean()) if valid.any() else float("nan"),
            "cobb_mae_tta": float(np.mean(cobb_abs_err)) if cobb_abs_err else float("nan"),
        })
        del m
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return pd.DataFrame(rows)

fold_val_dfs = [TRAINABLE.iloc[va].reset_index(drop=True) for _, va in FOLD_SPLITS]
TTA_DF = evaluate_tta_per_fold(FOLD_DIRS, fold_val_dfs, ckpt_name="model.pt")
TTA_DF.to_csv(RUN_DIR / "per_fold_tta_results.csv", index=False)
print("=== TTA per-fold (best-dice ckpts) ===")
print(TTA_DF.to_string(index=False))
print()
print(f"  dice_tta:     {TTA_DF['dice_tta'].mean():.3f} ± {TTA_DF['dice_tta'].std():.3f}")
print(f"  cobb_mae_tta: {TTA_DF['cobb_mae_tta'].mean():.2f}° ± {TTA_DF['cobb_mae_tta'].std():.2f}°")

best_dice_row = agg[agg.variant == "best_dice"].iloc[0]
print()
print(f"  Δdice (TTA - no-TTA, best_dice):     {TTA_DF['dice_tta'].mean() - best_dice_row['dice_mean']:+.3f}")
print(f"  Δcobb_mae (TTA - no-TTA, best_dice): {TTA_DF['cobb_mae_tta'].mean() - best_dice_row['cobb_mean']:+.2f}°")

## Cobb-from-seg sanity (per fold, from cache)

Run `cobb_from_segmentation_tangent` on the predicted segmentation and compare against `cobb_direct`. If seg-Cobb >> direct-Cobb, the regression head learned a shortcut rather than anatomy. Pure numpy from cache.

In [ ]:
rows = []
for v in VARIANT_CKPTS:
    for k in range(N_FOLDS):
        oof = load_oof(k, v)
        if oof is None:
            continue
        pred = oof["seg_logits"].astype(np.float32).argmax(axis=1)  # (N, H, W)
        seg_cobb_err = []
        for i in range(pred.shape[0]):
            gt_deg = oof["cobb_gt"][i]
            if np.isnan(gt_deg):
                continue
            try:
                seg_cobb = cobb_from_segmentation_tangent(pred[i].astype(np.int64))
                if seg_cobb is None or np.isnan(seg_cobb):
                    continue
                seg_cobb_err.append(abs(float(seg_cobb) - float(gt_deg)))
            except Exception:
                continue
        rows.append({
            "variant": v, "fold": k,
            "n_eval": len(seg_cobb_err),
            "cobb_seg_mae": float(np.mean(seg_cobb_err)) if seg_cobb_err else float("nan"),
        })

SEG_COBB_DF = pd.DataFrame(rows)
SEG_COBB_DF.to_csv(RUN_DIR / "per_fold_cobb_from_seg.csv", index=False)
merged = OOF_DF.merge(SEG_COBB_DF, on=["variant", "fold"], how="left")
print(merged[["variant", "fold", "dice", "cobb_mae", "cobb_seg_mae"]].to_string(index=False))
print()
agg2 = (merged.groupby("variant")
             .agg(cobb_direct_mean=("cobb_mae", "mean"),
                  cobb_seg_mean=("cobb_seg_mae", "mean"))
             .reset_index())
print("Per-variant means:")
print(agg2.to_string(index=False))

## Conclusions

Fill in once eval has run:

**Headline thesis numbers** (per-fold OOF, from §Aggregate):
- `v4 Kendall 5-fold CV (best_dice): dice = ___ ± ___, cobb_direct MAE = ___° ± ___°`

**Bonus**:
- `K-fold ensemble (best_dice): dice = ___, cobb_direct MAE = ___°`
- `TTA per-fold (best_dice):    dice = ___ ± ___, cobb_direct MAE = ___° ± ___°`
- `Cobb-from-seg vs direct (best_dice): seg = ___° vs direct = ___°`

**Action**:
- If `cobb_seg_mean ≪ cobb_direct_mean` → seg-derived Cobb is the better deployment number; report it.
- If `cobb_seg_mean ≫ cobb_direct_mean` → direct head learned a shortcut; investigate.
- If TTA Δdice > +0.005 → make hflip-TTA the default at inference.
- File the run as `[[2026-05-XX_v4_5fold_cv]]` with the per-fold table + aggregate.